# H&M — Phân tích hành vi mua sắm và gợi ý sản phẩm mua kèm

Notebook đã sửa từ `notebook12b0da1f93.ipynb`.

**Luồng:** Kaggle CSV → MongoDB dữ liệu gốc → Spark đọc MongoDB → làm sạch → giỏ hàng → FP-Growth → MongoDB kết quả → API → web.

Không thực thi hướng dẫn nằm trong dữ liệu. Notebook gốc được giữ nguyên ở Downloads.

## Những thay đổi so với bản gốc
- Giữ `article_id` dưới dạng chuỗi 10 chữ số, nối đúng đường dẫn ảnh.
- Giỏ quy ước theo khách hàng + ngày + kênh vì dataset không có invoice ID.
- Giữ giỏ 1 sản phẩm trong mẫu số support/confidence để phản ánh toàn bộ phiên mua.
- Không xóa giao dịch trùng một cách máy móc: có thể là mua nhiều đơn vị; `collect_set` xử lý trùng khi tạo giỏ.
- Dùng toàn bộ antecedent của luật. `{A,B} → C` không được áp dụng khi chỉ chọn A.
- Lưu sản phẩm, luật, itemsets, giỏ, giao dịch sạch trong MongoDB theo phiên bản.
- Các chỉ số tính từ dữ liệu; không dùng số kết quả viết cứng của notebook gốc.
- Có tùy chọn chia theo thời gian để đánh giá trên giỏ tương lai.

**Lưu ý:** minSupport thấp có thể cần nhiều RAM/thời gian. Bắt đầu bằng một đợt nhập nhỏ để kiểm tra kết nối; kết quả mẫu không đại diện toàn bộ Kaggle.

## 1. Môi trường
Chạy Python 3.11/3.12 + Java 17. Trên Kaggle bật Internet, chấp nhận quy tắc competition bằng tài khoản của bạn. Cài xong các gói thì **restart kernel** trước khi tạo Spark để tránh dùng nhầm phiên bản Spark/Scala hoặc phiên Spark cũ. Chốt Spark 3.5.3 + MongoDB Connector 10.4.1 / Scala 2.12. Docker trong repository là cách chạy ổn định trên Windows.

In [ ]:
%pip install pyspark==3.5.3 numpy==1.26.4 scipy==1.13.1 "setuptools>=68,<81" pymongo==4.10.1 python-dotenv==1.0.1 "kagglehub>=0.3.13,<1"

## 2. Đường dẫn dữ liệu & cấu hình
Nếu dataset đã được gắn vào Kaggle thì không tải lại. Nếu chạy ở máy cá nhân, dùng đúng API `kagglehub.competition_download` của bạn. Kaggle có thể yêu cầu đăng nhập/chấp nhận quy tắc; không ghi API key vào notebook.

In [ ]:
import os
from pathlib import Path
from argparse import Namespace
import getpass
from dotenv import load_dotenv
load_dotenv()

candidates = [Path('/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations'),
              Path('/kaggle/input/h-and-m-personalized-fashion-recommendations'), Path('data/raw')]
DATA_PATH = next((p for p in candidates if (p / 'articles.csv').exists()), None)
if DATA_PATH is None:
    import kagglehub
    DATA_PATH = Path(kagglehub.competition_download('h-and-m-personalized-fashion-recommendations'))
print('Data path:', DATA_PATH)

# Local: MongoDB đang chạy. Kaggle: dùng MongoDB Atlas với quyền/network phù hợp.
# Có thể đặt Kaggle Secret MONGODB_URI; tuyệt đối không in URI ra output.
if not os.getenv('MONGODB_URI'):
    if Path('/kaggle').exists():
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ['MONGODB_URI'] = UserSecretsClient().get_secret('MONGODB_URI')
        except Exception:
            os.environ['MONGODB_URI'] = getpass.getpass('MongoDB URI (ẩn): ')
    else:
        os.environ['MONGODB_URI'] = 'mongodb://localhost:27017'
os.environ.setdefault('MONGODB_DATABASE', 'hm_recommendation')
os.environ.setdefault('SPARK_DRIVER_MEMORY', '4g')
os.environ.setdefault('SPARK_PARTITIONS', '64')
os.environ.setdefault('SPARK_MASTER', 'local[4]')

IMPORT_LIMIT = 100000  # Chạy thử; đặt 0 để nhập toàn bộ giao dịch
MIN_SUPPORT = 0.001   # Chạy thử; khi chạy đầy đủ có thể thử 0.0001
MIN_CONFIDENCE = 0.3
MIN_LIFT = 1.0
CUTOFF = None        # Full data: '2020-09-15' để giữ lại giỏ tương lai


## 3. Nhập CSV vào MongoDB
Đọc theo batch 5.000 dòng, không tải toàn bộ giao dịch vào RAM. Các collection nguồn mới chỉ được kích hoạt sau khi nhập đầy đủ. Nhập lại sẽ tạo đợt mới, không ghi đè nguồn đang dùng. Giữ ID chuỗi ngay từ lúc đọc CSV.

In [ ]:
"""Bounded-memory CSV import. Publish source manifest only after all imports succeed."""
import argparse
import csv
import os
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from pymongo import MongoClient, InsertOne
from dotenv import load_dotenv

def import_dataset(args):
    if args.limit < 0:
        raise ValueError('limit must be >= 0')
    root = Path(args.data_path)
    files = {'articles': 'articles.csv', 'customers': 'customers.csv', 'transactions': 'transactions_train.csv'}
    for filename in files.values():
        if not (root / filename).is_file():
            raise FileNotFoundError(root / filename)
    with MongoClient(os.getenv('MONGODB_URI', 'mongodb://localhost:27017')) as client:
        db = client[os.getenv('MONGODB_DATABASE', 'hm_recommendation')]
        run = uuid4().hex
        collections = {key: f'raw_{key}_{run}' for key in files}
        counts = {}
        for key, filename in files.items():
            coll = db[collections[key]]
            count, batch = 0, []
            with (root / filename).open(encoding='utf-8-sig', newline='') as stream:
                for row in csv.DictReader(stream):
                    if key == 'transactions' and args.limit and count >= args.limit:
                        break
                    # Keep exact duplicate rows: they can represent multiple units.
                    # CSV is read as strings so leading zeros are never discarded.
                    row['_id'] = count
                    batch.append(InsertOne(row))
                    count += 1
                    if len(batch) >= 5000:
                        coll.bulk_write(batch, ordered=False)
                        batch.clear()
                        if count % 500000 == 0:
                            print(f'{key}: {count:,}', flush=True)
                if batch:
                    coll.bulk_write(batch, ordered=False)
            counts[key] = count
            coll.create_index('article_id' if key == 'articles' else 'customer_id')
        db[collections['transactions']].create_index([('t_dat', 1), ('sales_channel_id', 1)])
        db.metadata.replace_one({'_id':'active_source'}, {'_id':'active_source', 'run_id':run, 'collections':collections, 'counts':counts, 'sample':bool(args.limit), 'imported_at':datetime.now(timezone.utc).isoformat()}, upsert=True)
        print({'source_run':run, 'counts':counts, 'sample':bool(args.limit)})

def main():
    load_dotenv()
    parser = argparse.ArgumentParser()
    parser.add_argument('--data-path', default=os.getenv('DATA_PATH', 'data/raw'))
    parser.add_argument('--limit', type=int, default=0, help='0 = full data; positive = first N transaction rows for a smoke test only')
    import_dataset(parser.parse_args())



In [ ]:
import_dataset(Namespace(data_path=str(DATA_PATH), limit=IMPORT_LIMIT))


## 4. Spark làm sạch, tạo giỏ, huấn luyện, đánh giá & lưu kết quả
Đây là cùng mã nguồn với `training/pipeline.py`; không có mô hình thứ hai khác với API. Dữ liệu huấn luyện đọc trực tiếp từ MongoDB qua Spark Connector. Chỉ công bố mô hình khi đã ghi xong tất cả collection và index.

- `support(X→Y) = số giỏ chứa X∪Y / số giỏ huấn luyện`.
- `confidence(X→Y) = số giỏ chứa X∪Y / số giỏ chứa X`.
- `lift = confidence / support(Y)`; chỉ giữ `lift > MIN_LIFT`.
- Nếu có `CUTOFF`: không đưa giao dịch sau cutoff vào huấn luyện, giá trung bình hay độ phổ biến.
- Đánh giá: tối đa 10.000 giỏ tương lai, ẩn một sản phẩm theo hash + seed 42, gợi ý top 6 từ các sản phẩm còn lại. Đây là đánh giá masked-item xác định, không phải đánh giá triển khai thực tế. Precision@6 dùng mẫu số 6, Recall@6 bằng hit rate vì mỗi giỏ ẩn đúng một item. Coverage là tỷ lệ giỏ có ít nhất một gợi ý. Chỉ đánh giá giỏ có ít nhất 2 sản phẩm.

Không so sánh trực tiếp với số liệu cũ vì quy ước giỏ và mẫu số đã thay đổi.

In [ ]:
"""MongoDB -> Spark -> baskets -> FP-Growth -> versioned MongoDB outputs."""
import argparse
import os
import sys
import logging
from datetime import date, datetime, timezone
from uuid import uuid4
from dotenv import load_dotenv
from pymongo import MongoClient
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.ml.fpm import FPGrowth
from pyspark import StorageLevel

CONNECTOR = 'org.mongodb.spark:mongo-spark-connector_2.12:10.4.1'

def clean_transactions(df, article_ids):
    return (df.filter(F.trim('article_id').rlike('^[0-9]{1,10}$')).select(F.to_date('t_dat').alias('t_dat'), F.trim('customer_id').alias('customer_id'),
                      F.lpad(F.trim('article_id'), 10, '0').alias('article_id'),
                      F.col('price').cast('double').alias('price'), F.col('sales_channel_id').cast('int').alias('sales_channel_id'))
        .filter(F.col('t_dat').isNotNull() & (F.length('customer_id') > 0) & F.col('article_id').rlike('^[0-9]{10}$')
                & F.col('price').isNotNull() & ~F.isnan('price') & (F.col('price') > 0)
                & (F.col('price') < float('inf')) & F.col('sales_channel_id').isin(1, 2))
        .join(F.broadcast(article_ids), 'article_id', 'left_semi'))

def make_baskets(transactions):
    return (transactions.groupBy('customer_id', 't_dat', 'sales_channel_id')
        .agg(F.sort_array(F.collect_set('article_id')).alias('items'))
        .withColumn('basket_size', F.size('items')))

def group_baskets(sku_baskets, articles):
    expanded = sku_baskets.withColumn('article_id', F.explode('items')).join(
        F.broadcast(articles.select('article_id','product_code')), 'article_id')
    return (expanded.groupBy('customer_id','t_dat','sales_channel_id')
        .agg(F.sort_array(F.collect_set('product_code')).alias('items'))
        .withColumn('basket_size',F.size('items')))

def parse_sweep_configs(value=None):
    if not value:
        return [(0.0001,0.30),(0.0001,0.15),(0.00005,0.15)]
    choices=[]
    for pair in value.split(','):
        support,confidence = map(float,pair.split(':'))
        if not 0 < support <= 1 or not 0 <= confidence <= 1:
            raise ValueError('Sweep support must be in (0,1], confidence in [0,1]')
        if (support,confidence) not in choices:
            choices.append((support,confidence))
    return choices

def prepare_cases(baskets, limit=10000, seed=42):
    # Stable hash order masks different items instead of always the greatest ID.
    test = (baskets.filter(F.col('basket_size') >= 2)
        .withColumn('key', F.sha2(F.concat_ws('|', 'customer_id', F.col('t_dat').cast('string'), F.col('sales_channel_id').cast('string')), 256))
        .withColumn('sample_order', F.xxhash64('key', F.lit(seed)))
        .orderBy('sample_order', 'key').limit(limit)
        .withColumn('masked_order', F.sort_array(F.transform('items', lambda item:
            F.struct(F.xxhash64('key', item, F.lit(seed)).alias('hash'), item.alias('item')))))
        .withColumn('target', F.element_at('masked_order', 1)['item'])
        .withColumn('context', F.array_except('items', F.array('target'))))
    return test.select('key', 'target', 'context')

def evaluate(baskets, rules, limit=10000, seed=42, popularity=None, cases=None):
    owns_cases = cases is None
    test = cases if cases is not None else prepare_cases(baskets, limit, seed).persist(StorageLevel.DISK_ONLY)
    total = test.count()
    def scores(predictions):
        result = predictions.agg(F.countDistinct('key').alias('covered'),
            F.sum(F.when(F.col('candidate') == F.col('target'), 1).otherwise(0)).alias('hits')).first()
        hits = result['hits'] or 0
        return {'precision_at_6':hits/(6*total), 'recall_at_6':hits/total,
                'coverage':result['covered']/total, 'hits':hits}
    base = {'baskets':total, 'protocol':'temporal_seeded_masked_item', 'seed':seed,
            'sample_limit':limit, 'k':6}
    if total == 0:
        if owns_cases:
            test.unpersist()
        return {**base,'precision_at_6':0.0,'recall_at_6':0.0,'coverage':0.0,'status':'no_eligible_baskets'}
    keyed_rules = rules.withColumn('join_id', F.element_at('antecedent', 1))
    matches = (test.withColumn('join_id', F.explode('context')).join(keyed_rules, 'join_id')
        .filter(F.size(F.array_except('antecedent', 'context')) == 0)
        .withColumn('candidate', F.explode('consequent'))
        .filter(~F.array_contains('context', F.col('candidate'))))
    best = (matches.withColumn('rn', F.row_number().over(Window.partitionBy('key','candidate').orderBy(F.desc('confidence'),F.desc('lift'),F.desc('support'))))
        .filter('rn = 1').drop('rn')
        .withColumn('rank', F.row_number().over(Window.partitionBy('key').orderBy(F.desc('confidence'),F.desc('lift'),F.desc('support'),'candidate')))
        .filter('rank <= 6'))
    best = best.select('key','target','candidate').persist(StorageLevel.DISK_ONLY)
    try:
        result = {**base, **scores(best)}
        if popularity is not None:
            # Six plus the largest context guarantees enough eligible candidates.
            max_context = test.agg(F.max(F.size('context'))).first()[0] or 0
            pool = popularity.orderBy(F.desc('count'),'candidate').limit(6+max_context)
            popular = (test.crossJoin(F.broadcast(pool))
                .filter(~F.array_contains('context', F.col('candidate')))
                .withColumn('rank', F.row_number().over(Window.partitionBy('key').orderBy(F.desc('count'),'candidate')))
                .filter('rank <= 6').select('key','target','candidate'))
            result['popularity'] = scores(popular)
            # Runtime fallback applies only when no rule can be rendered.
            fallback = popular.join(best.select('key').distinct(), 'key', 'left_anti')
            result['hybrid'] = scores(best.unionByName(fallback))
        return result
    finally:
        best.unpersist()
        if owns_cases:
            test.unpersist()

def temporal_split(baskets, train_end, validation_end):
    if date.fromisoformat(train_end) >= date.fromisoformat(validation_end):
        raise ValueError('train_end must precede validation_end')
    return (baskets.filter(F.col('t_dat') <= F.lit(train_end)),
            baskets.filter((F.col('t_dat') > F.lit(train_end)) & (F.col('t_dat') <= F.lit(validation_end))),
            baskets.filter(F.col('t_dat') > F.lit(validation_end)))

def selection_key(row):
    # Test metrics and rule counts never participate in model selection.
    score = row['validation']
    return (score['recall_at_6'], score['precision_at_6'], score['coverage'],
            row['min_support'], row['min_confidence'])

def run(args):
    uri = os.getenv('MONGODB_URI', 'mongodb://localhost:27017')
    database = os.getenv('MONGODB_DATABASE', 'hm_recommendation')
    client = MongoClient(uri)
    db = client[database]
    source = db.metadata.find_one({'_id':'active_source'})
    previous = db.metadata.find_one({'_id':'active_model'})
    item_level = getattr(args, 'item_level', 'article_id')
    seed = getattr(args, 'seed', 42)
    evaluation_limit = getattr(args, 'evaluation_limit', 10000)
    validation_end = getattr(args, 'validation_end', None)
    reuse = (not getattr(args, 'no_reuse', False) and previous
             and previous.get('source_run') == (source or {}).get('run_id')
             and previous.get('basket_definition') == 'customer_id + t_dat + sales_channel_id; include singletons')
    if not source:
        raise RuntimeError('Run python -m training.import_data first.')
    master = os.getenv('SPARK_MASTER', 'local[4]')
    partitions = int(os.getenv('SPARK_PARTITIONS', '64'))
    if partitions < 1:
        raise ValueError('SPARK_PARTITIONS must be positive')
    os.environ.setdefault('PYSPARK_PYTHON', sys.executable)
    builder = (SparkSession.builder.appName('HM_MongoDB_FP_Growth')
        .master(master)
        .config('spark.jars.packages', CONNECTOR)
        .config('spark.driver.memory', os.getenv('SPARK_DRIVER_MEMORY', '4g'))
        .config('spark.sql.shuffle.partitions', str(partitions))
        .config('spark.sql.adaptive.coalescePartitions.enabled', 'false')
        .config('spark.local.dir', os.getenv('SPARK_LOCAL_DIR', '/tmp/hm-spark') if os.name != 'nt' else os.getenv('SPARK_LOCAL_DIR', os.path.join(os.getenv('TEMP', '.'), 'hm-spark')))
        .config('spark.mongodb.read.connection.uri', uri)
        .config('spark.mongodb.write.connection.uri', uri))
    if master.startswith('local'):
        # Windows hostnames may contain underscores and are not valid Spark URLs.
        builder = builder.config('spark.driver.host','127.0.0.1').config('spark.driver.bindAddress','127.0.0.1')
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel('WARN')
    run_id = uuid4().hex
    outputs = {key:f'{key}_{run_id}' for key in ('products','rules','itemsets','baskets','clean_transactions')}
    def read(key):
        # Explicit strings prevent Mongo inference from losing zero-prefixed IDs.
        schemas = {
            'articles':'article_id string, product_code string, prod_name string, product_type_name string, product_group_name string, colour_group_name string, detail_desc string',
            'transactions':'t_dat string, customer_id string, article_id string, price string, sales_channel_id string'}
        return (spark.read.format('mongodb').option('database', database).option('collection',source['collections'][key]).schema(schemas[key]).load())
    def write(df, key):
        (df.write.format('mongodb').mode('overwrite').option('database',database).option('collection',outputs[key]).save())
    persisted = []
    def disk_cache(df):
        persisted.append(df.persist(StorageLevel.DISK_ONLY))
        return persisted[-1]
    def progress(message):
        print(f'[{datetime.now(timezone.utc).isoformat()}] {message}', flush=True)
    try:
        progress(f'run={run_id}; source={source["run_id"]}; master={master}; partitions={partitions}; heap={os.getenv("SPARK_DRIVER_MEMORY", "4g")}')
        articles = (read('articles').filter(F.trim('article_id').rlike('^[0-9]{1,10}$')).withColumn('article_id', F.lpad(F.trim('article_id'),10,'0'))
            .withColumn('product_code', F.when(F.trim('product_code').rlike('^[0-9]{1,7}$'), F.lpad(F.trim('product_code'),7,'0'))
                        .otherwise(F.substring('article_id',1,7)))
            .filter(F.col('article_id').rlike('^[0-9]{10}$')).dropDuplicates(['article_id']).persist(StorageLevel.MEMORY_AND_DISK))
        persisted.append(articles)
        progress('Cleaning all MongoDB transactions; persisting large frames on disk')
        if reuse:
            progress('Reusing complete cleaned transactions and SKU baskets from the same source run')
            for key in ('clean_transactions', 'baskets'):
                outputs[key] = previous['collections'][key]
            def read_saved(key, schema):
                return spark.read.format('mongodb').option('database',database).option('collection',outputs[key]).schema(schema).load()
            transactions = disk_cache(read_saved('clean_transactions','t_dat date, customer_id string, article_id string, price double, sales_channel_id int'))
            sku_baskets = disk_cache(read_saved('baskets','t_dat date, customer_id string, sales_channel_id int, items array<string>, basket_size int'))
            transaction_count = previous['transactions']
        else:
            transactions = disk_cache(clean_transactions(read('transactions'), articles.select('article_id')))
            transaction_count = transactions.count()
            sku_baskets = disk_cache(make_baskets(transactions))
        progress(f'Clean transactions: {transaction_count:,}; building baskets')
        baskets = sku_baskets
        if item_level == 'product_code':
            progress('Grouping SKU variants into product families; deduplicating each family within a basket')
            baskets = disk_cache(group_baskets(sku_baskets,articles))
        train_transactions = transactions.filter(F.col('t_dat') <= F.lit(args.cutoff)) if args.cutoff else transactions
        train_baskets = baskets.filter(F.col('t_dat') <= F.lit(args.cutoff)) if args.cutoff else baskets
        # Include singleton baskets: support/confidence describe all shopping sessions.
        # The original notebook excluded these, inflating conditional rates.
        # Reuse persisted baskets rather than caching a duplicate items-only copy.
        train = train_baskets.select('items')
        basket_count = train.count()
        progress(f'Training baskets: {basket_count:,}; fitting FP-Growth')
        if basket_count == 0:
            raise ValueError('No training baskets. Check data and --cutoff.')
        popularity = disk_cache(train.select(F.explode('items').alias('candidate')).groupBy('candidate').count())
        validation_results = []
        if validation_end:
            _, validation_baskets, test_baskets = temporal_split(baskets, args.cutoff, validation_end)
            validation_cases = disk_cache(prepare_cases(validation_baskets, evaluation_limit, seed))
            if validation_cases.count() == 0:
                raise ValueError('Validation has no multi-item baskets; choose different dates.')
            choices = parse_sweep_configs(getattr(args,'sweep_configs',None))
            models = {}
            rules_by_support = {}
            for support in sorted({s for s,c in choices}, reverse=True):
                confidence = min(c for s,c in choices if s == support)
                progress(f'Validation sweep: support={support}, confidence>={confidence}')
                candidate_model = FPGrowth(itemsCol='items', minSupport=support, minConfidence=confidence, numPartitions=partitions).fit(train)
                models[support] = candidate_model
                # freqItemsets is lazy: persist it before generating rules, or
                # rule counts/export can mine the complete dataset repeatedly.
                disk_cache(candidate_model.freqItemsets)
                all_rules = disk_cache(candidate_model.associationRules.filter(F.col('lift') > args.min_lift))
                rules_by_support[support] = all_rules
                for s,c in choices:
                    if s != support:
                        continue
                    candidate_rules = all_rules.filter(F.col('confidence') >= c)
                    validation_results.append({'min_support':s, 'min_confidence':c,
                        'rules':candidate_rules.count(), 'validation':evaluate(validation_baskets,candidate_rules,
                        evaluation_limit,seed,popularity,validation_cases)})
                    row = validation_results[-1]
                    progress(f'Validation complete: support={s}, confidence={c}, rules={row["rules"]}, recall@6={row["validation"]["recall_at_6"]:.6f}, coverage={row["validation"]["coverage"]:.6f}')
            selected = max(validation_results,key=selection_key)
            args.min_support, args.min_confidence = selected['min_support'], selected['min_confidence']
            model = models[args.min_support]
            base_rules = rules_by_support[args.min_support]
            progress(f'Selected using validation only: support={args.min_support}, confidence={args.min_confidence}')
        else:
            model = FPGrowth(itemsCol='items', minSupport=args.min_support, minConfidence=args.min_confidence, numPartitions=partitions).fit(train)
            disk_cache(model.freqItemsets)
            base_rules = model.associationRules
            test_baskets = baskets.filter(F.col('t_dat') > F.lit(args.cutoff)) if args.cutoff else None
        rules = disk_cache(base_rules.filter((F.col('lift') > args.min_lift) & (F.col('confidence') >= args.min_confidence)))
        progress('FP-Growth fitted; evaluating and saving versioned results')
        evaluation = evaluate(test_baskets, rules, evaluation_limit, seed, popularity) if args.cutoff else None
        stats = train_transactions.groupBy('article_id').agg(F.count('*').alias('purchase_count'), F.avg('price').alias('price_normalized'))
        popularity_stats = popularity.withColumnRenamed('candidate',item_level).withColumnRenamed('count','recommendation_popularity')
        products = (articles.join(stats,'article_id','left').join(popularity_stats,item_level,'left').select('article_id','product_code',
            F.coalesce('prod_name', 'article_id').alias('name'),
            F.coalesce('product_group_name', F.lit('Other')).alias('category'),
            F.col('product_type_name').alias('product_type'), F.col('colour_group_name').alias('colour'),
            F.col('detail_desc').alias('description'), F.coalesce('purchase_count',F.lit(0)).alias('purchase_count'), 'price_normalized',
            F.coalesce('recommendation_popularity',F.lit(0)).alias('recommendation_popularity'),
            F.concat(F.lit('/images/'),F.substring('article_id',1,3),F.lit('/'),F.col('article_id'),F.lit('.jpg')).alias('image_url')))
        if not reuse:
            write(transactions, 'clean_transactions')
            write(sku_baskets, 'baskets')
        write(products, 'products')
        write(rules, 'rules')
        write(model.freqItemsets.withColumn('support',F.col('freq')/basket_count), 'itemsets')
        db[outputs['products']].create_index('article_id',unique=True)
        db[outputs['products']].create_index([('category',1),('purchase_count',-1)])
        db[outputs['products']].create_index([('recommendation_popularity',-1),(item_level,1)])
        db[outputs['products']].create_index([('product_code',1),('purchase_count',-1),('article_id',1)])
        db[outputs['rules']].create_index('antecedent')
        top_categories = (train_transactions.join(articles.select('article_id','product_group_name'),'article_id')
            .groupBy('product_group_name').count().orderBy(F.desc('count')).limit(10).collect())
        dates = train_transactions.agg(F.min('t_dat').alias('start'),F.max('t_dat').alias('end')).first()
        manifest = {'_id':'active_model', 'run_id':run_id, 'source_run':source['run_id'], 'sample':source['sample'],
            'trained_at':datetime.now(timezone.utc).isoformat(), 'collections':outputs,
            'transactions':transaction_count, 'training_transactions':train_transactions.count(),
            'baskets':basket_count, 'products':products.count(), 'rules':rules.count(), 'itemsets':model.freqItemsets.count(),
            'date_range':f"{dates['start']} → {dates['end']}",
            'parameters':{'min_support':args.min_support,'min_confidence':args.min_confidence,'min_lift':args.min_lift},
            'evaluation':evaluation, 'item_level':item_level,
            'validation':{'end':validation_end,'results':validation_results,'selected_by':'FP-Growth recall, precision, coverage on validation only'} if validation_end else None,
            'top_categories':[{'category':r['product_group_name'] or 'Other','count':r['count']} for r in top_categories],
            'basket_definition':'customer_id + t_dat + sales_channel_id; include singletons', 'cutoff':args.cutoff,
            'execution':{'master':master,'partitions':partitions,'large_frame_storage':'DISK_ONLY','reused_clean_data':bool(reuse)}}
        # Atomic pointer update: API never mixes products and rules from different runs.
        db.model_runs.insert_one({**manifest,'_id':run_id})
        if not getattr(args, 'no_activate', False):
            db.metadata.replace_one({'_id':'active_model'},manifest,upsert=True)
        print({k:v for k,v in manifest.items() if k not in ('collections','top_categories')})
        return manifest
    finally:
        # A dead JVM must not hide the original stage failure with a stop() error.
        try:
            for df in persisted:
                df.unpersist()
            spark.stop()
        except Exception:
            logging.warning('Spark cleanup failed; JVM may have already stopped. See the first error above.')
        finally:
            client.close()

def main():
    load_dotenv()
    parser=argparse.ArgumentParser()
    parser.add_argument('--min-support',type=float,default=float(os.getenv('MIN_SUPPORT','0.0001')))
    parser.add_argument('--min-confidence',type=float,default=float(os.getenv('MIN_CONFIDENCE','0.3')))
    parser.add_argument('--min-lift',type=float,default=float(os.getenv('MIN_LIFT','1')))
    parser.add_argument('--cutoff',help='YYYY-MM-DD; train on/before date, evaluate on later baskets')
    parser.add_argument('--validation-end',help='Sweep thresholds on (cutoff, validation-end]; evaluate selected model only after validation-end')
    parser.add_argument('--sweep-configs',help='Comma-separated support:confidence pairs; requires --validation-end')
    parser.add_argument('--seed',type=int,default=42)
    parser.add_argument('--evaluation-limit',type=int,default=10000)
    parser.add_argument('--item-level',choices=['article_id','product_code'],default='article_id')
    parser.add_argument('--no-reuse',action='store_true',help='Rebuild clean data instead of reusing the same source run')
    parser.add_argument('--no-activate',action='store_true',help='Save a candidate without replacing the web model')
    args=parser.parse_args()
    if not 0 < args.min_support <= 1 or not 0 <= args.min_confidence <= 1 or args.min_lift < 0:
        parser.error('Invalid support/confidence/lift')
    if args.cutoff:
        date.fromisoformat(args.cutoff)
    if args.evaluation_limit < 1:
        parser.error('evaluation-limit must be positive')
    if args.validation_end:
        if not args.cutoff or date.fromisoformat(args.cutoff) >= date.fromisoformat(args.validation_end):
            parser.error('--validation-end requires an earlier --cutoff')
    if args.sweep_configs:
        if not args.validation_end:
            parser.error('--sweep-configs requires --validation-end')
        try:
            parse_sweep_configs(args.sweep_configs)
        except ValueError as exc:
            parser.error(str(exc))
    run(args)



In [ ]:
result = run(Namespace(min_support=MIN_SUPPORT, min_confidence=MIN_CONFIDENCE,
                       min_lift=MIN_LIFT, cutoff=CUTOFF, seed=42, item_level="article_id",
                       validation_end=None, evaluation_limit=10000))


Có thể đặt validation_end="2020-09-15", cutoff="2020-09-08" trong Namespace để thử ba cấu hình và chỉ đánh giá cấu hình được chọn trên các ngày sau validation_end. Baseline popularity và hybrid đều chỉ dùng tần suất giỏ huấn luyện. Đặt item_level="product_code" để khai phá ở cấp mẫu sản phẩm; chỉ số ở cấp mẫu không so trực tiếp với cấp SKU.

## 5. Kiểm tra sản phẩm & luật gợi ý từ MongoDB
Khớp đầy đủ vế điều kiện, bỏ sản phẩm đã chọn, lấy luật mạnh nhất cho mỗi sản phẩm và xếp confidence → lift → support. Nếu không có luật, API trả sản phẩm phổ biến ở trường suggestions riêng, hiển thị “Có thể bạn thích”. Không gắn nhãn mua kèm hoặc confidence/lift cho phần dự phòng.

In [ ]:
"""Rule matching shared by the MongoDB API, demo, and tests."""
def rank_recommendations(rules, items, limit=6):
    basket = set(items)
    best = {}
    for rule in rules:
        antecedent = set(rule.get('antecedent', []))
        if not antecedent or not antecedent.issubset(basket):
            continue
        for article_id in set(rule.get('consequent', [])) - basket:
            candidate = {key: rule[key] for key in ('confidence', 'lift', 'support')}
            candidate.update(article_id=article_id, antecedent=sorted(antecedent))
            rank = (candidate['confidence'], candidate['lift'], candidate['support'])
            old = best.get(article_id)
            if old is None or rank > (old['confidence'], old['lift'], old['support']):
                best[article_id] = candidate
    return sorted(best.values(), key=lambda r: (-r['confidence'], -r['lift'], -r['support'], r['article_id']))[:limit]


In [ ]:
from pymongo import MongoClient
client = MongoClient(os.environ['MONGODB_URI'])
db = client[os.environ['MONGODB_DATABASE']]
manifest = db.metadata.find_one({'_id': 'active_model'})
rules_collection = db[manifest['collections']['rules']]
products_collection = db[manifest['collections']['products']]
first_rule = rules_collection.find_one()
if first_rule:
    basket_items = first_rule['antecedent']
    matched = rules_collection.find({'antecedent': {'$in': basket_items}}, {'_id': 0})
    for rec in rank_recommendations(matched, basket_items):
        p = products_collection.find_one({'article_id': rec['article_id']})
        print(p['name'] if p else rec['article_id'], rec)
else:
    print('Không có luật đạt ngưỡng. Điều chỉnh ngưỡng hoặc dùng thêm dữ liệu.')
client.close()


## 6. Hiển thị trên web
Trên máy cá nhân, đặt `.env`: `APP_MODE=mongo`, URI và tên DB khớp notebook. Thư mục `DATA_PATH/images` phải chứa ảnh Kaggle. Chạy `python -m backend.app`, mở http://127.0.0.1:8000. Web đọc collection từ manifest `active_model` để giữ sản phẩm và luật cùng phiên bản.

Kaggle không phải nơi host web lâu dài. Nếu huấn luyện trên Kaggle/Atlas, web máy cá nhân cần kết nối cùng MongoDB và có ảnh tải về máy. Số liệu khách hàng không được đưa ra API công khai.

Tham khảo: [H&M Kaggle](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/data), [Spark FP-Growth](https://spark.apache.org/docs/3.5.3/ml-frequent-pattern-mining.html), [MongoDB Spark Connector 10.4](https://www.mongodb.com/docs/spark-connector/v10.4/).